© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

## Setup

In [1]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib
from ood_metrics import fpr_at_95_tpr
from sklearn.metrics import average_precision_score

torch.cuda.memory.empty_cache()

seed_everything(0, verbose=False)

device = 0  # GPU 0 is already the right one
img_idx = 10  # TODO: change to the index of the image you want to visualize
config_path = "configs\dinov2\cityscapes\semantic\eomt_large_1024.yaml"
data_path = r"C:\Users\edgar\Documents\segmentation_project\code\anomaly_segmentation\ValidationDatasets"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)


def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array([plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))])
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping

def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

C:\Users\edgar\Documents\segmentation_project\code\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
).setup()

## Load model

In [3]:
# ── CELL: Load model (no Cityscapes data loader required) ─────────────────────
#
# img_size is read directly from the config, so you can skip "Load dataset"
# entirely when running on anomaly benchmarks.

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Read img_size straight from the config instead of from `data`
img_size = (1024, 1024)  # CityscapesSemantic default
num_classes = 19          # CityscapesSemantic default

# Load encoder
encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=num_classes,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
if "stuff_classes" in config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=img_size,
        num_classes=num_classes,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

## Load pre-trained weights from Hugging Face Hub
The model weights are downloaded from the Hugging Face Hub using the logger name from the config. Make sure you have a working internet connection.

In [4]:
name = config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")

if name is None:
    warnings.warn("No logger name found in the config. Please specify a model name.")
else:
    try:
        state_dict_path = hf_hub_download(
            repo_id=f"tue-mps/{name}",
            filename="pytorch_model.bin",
        )

        is_dinov3 = "dinov3" in name

        if is_dinov3:
            model_kwargs["ckpt_path"] = state_dict_path
            model_kwargs["delta_weights"] = True

        model = (
            lit_cls(
                img_size=img_size,
                num_classes=num_classes,
                network=network,
                **model_kwargs,
            )
            .eval()
            .to(device)
        )

        if not is_dinov3:
            state_dict = torch.load(
                state_dict_path, map_location=f"cuda:{device}", weights_only=True
            )
            model.load_state_dict(state_dict, strict=False)

    except RepositoryNotFoundError:
        warnings.warn(
            f"Pre-trained model not found for `{name}`. Please load your own checkpoint."
        )

## Semantic inference (pixel-wise classification)

> This inference method also works when applied to a model trained for panoptic segmentation.

Semantic inference computes per-pixel class scores by combining mask and class predictions:

$$
\sum_i p_i(c) \cdot m_i[h, w]
$$

Here, $p_i(c)$ is the class probability for class $c$ (excluding "no object"), and $m_i[h, w]$ is the sigmoid-normalized mask value for query $i$ at pixel $(h, w)$. The final class is selected by taking the argmax over classes.  
  
*This inference method was originally introduced in MaskFormer.*

In [7]:
# ── CELL: OoD evaluation on anomaly benchmarks ────────────────────────────────
#
# Prerequisites: Setup cell + Load model cell + Load weights cell must have run.
# The "Load dataset" (Cityscapes) cell can be SKIPPED entirely.
#
# Place anomaly_datasets.py next to this notebook before running.

from anomaly_datasets import get_dataset
import numpy as np
import torch

IGNORE_INDEX = 255

# ── Helper functions ───────────────────────────────────────────────────────────

def infer_semantic(img):
    """
    Run EoMT semantic inference on a single image tensor.

    Args:
        img : torch.Tensor [3, H, W]  — already normalised (from anomaly_datasets)

    Returns:
        logits : torch.Tensor [1, C, H_orig, W_orig]
        pred   : np.ndarray  [H_orig, W_orig]  argmax class prediction
    """
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], img_size, mode="bilinear"   # img_size from Load model cell
        )
        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        pred = logits[0].argmax(0).cpu().numpy()

    return logits, pred


def calculate_msp(logits_np):
    """Max Softmax Probability — HIGH means confident (InD)."""
    logits_shifted = logits_np - np.max(logits_np, axis=0, keepdims=True)
    exp = np.exp(logits_shifted)
    softmax = exp / np.sum(exp, axis=0, keepdims=True)
    return np.max(softmax, axis=0)


def calculate_entropy(logits_np):
    logits_shifted = logits_np - np.max(logits_np, axis=0, keepdims=True)
    exp = np.exp(logits_shifted)
    softmax = exp / np.sum(exp, axis=0, keepdims=True)
    log_probs = np.log(softmax + 1e-10)
    return np.sum(-softmax * log_probs, axis=0)


def calculate_max_logit(logits_np):
    """Negative max logit — HIGH means uncertain (OoD)."""
    return -np.max(logits_np, axis=0)


def calculate_metrics(scores, ood_mask, ind_mask):
    """AUPRC and FPR@95TPR. Expects HIGH score = OoD."""
    ood_out = scores[ood_mask]
    ind_out = scores[ind_mask]
    val_out   = np.concatenate((ind_out, ood_out))
    val_label = np.concatenate((np.zeros(len(ind_out)), np.ones(len(ood_out))))
    prc_auc = average_precision_score(val_label, val_out)
    fpr     = fpr_at_95_tpr(val_out, val_label)
    return prc_auc, fpr

#def calculate_rba_from_combined_logits(logits_np):
#    """Simplified RbA from already-combined per-pixel logits [C, H, W]."""
#    return -np.sum(logits_np, axis=0)

def calculate_rba_from_combined_logits(logits_np):
    # σ = tanh, not softmax or sigmoid
    return -np.sum(np.tanh(logits_np), axis=0)

# ── Datasets to evaluate ───────────────────────────────────────────────────────

DATASET_NAMES = [
    "RoadAnomaly",
    "RoadAnomaly21",
    "RoadObstacle21",
    "fs_static",
    "FS_LostFound",
]

# ── Main loop ──────────────────────────────────────────────────────────────────

for ds_name in DATASET_NAMES:
    print(f"\n{'='*60}")
    print(f"  Dataset: {ds_name}")
    print(f"{'='*60}")

    dataset = get_dataset(ds_name, root=data_path)

    msp_list, entropy_list, max_logit_list, ood_gts_list = [], [], [], []
    rba_list = []
    skipped = 0

    for idx in range(len(dataset)):
        torch.cuda.empty_cache()
        img, target = dataset[idx]          # target already in {0, 1, 255}
        img = (img * 255).byte()
        ood_gt = target.numpy()

        # Skip images with no OoD pixels (some FS L&F images are fully InD)
        if 1 not in np.unique(ood_gt):
            skipped += 1
            continue

        logits, pred = infer_semantic(img)
        logits_np = logits[0].cpu().numpy()     # [C, H, W]

        msp_list.append(calculate_msp(logits_np))
        entropy_list.append(calculate_entropy(logits_np))
        max_logit_list.append(calculate_max_logit(logits_np))
        rba_list.append(calculate_rba_from_combined_logits(logits_np))
        ood_gts_list.append(ood_gt)

    print(f"  Evaluated: {len(ood_gts_list)} images  |  Skipped (no OoD): {skipped}")

    if len(ood_gts_list) == 0:
        print("  ⚠  No valid images — check folder names in anomaly_datasets.py")
        continue

    ood_gts_array = np.array(ood_gts_list)
    ignore_mask   = (ood_gts_array == IGNORE_INDEX)
    ood_mask      = (ood_gts_array == 1) & ~ignore_mask
    ind_mask      = (ood_gts_array == 0) & ~ignore_mask

    # MSP is high for InD → negate so that HIGH = OoD for all three methods
    msp_auc,       msp_fpr       = calculate_metrics(-np.array(msp_list),       ood_mask, ind_mask)
    entropy_auc,   entropy_fpr   = calculate_metrics( np.array(entropy_list),   ood_mask, ind_mask)
    max_logit_auc, max_logit_fpr = calculate_metrics( np.array(max_logit_list), ood_mask, ind_mask)
    rba_auc,       rba_fpr       = calculate_metrics( np.array(rba_list),       ood_mask, ind_mask)

    print(f"\n  {'Method':<15} {'AUPRC':>10}  {'FPR@95TPR':>10}")
    print(f"  {'-'*38}")
    print(f"  {'MSP':<15} {msp_auc*100:>9.2f}%  {msp_fpr*100:>9.2f}%")
    print(f"  {'Entropy':<15} {entropy_auc*100:>9.2f}%  {entropy_fpr*100:>9.2f}%")
    print(f"  {'MaxLogit':<15} {max_logit_auc*100:>9.2f}%  {max_logit_fpr*100:>9.2f}%")
    print(f"  {'RbA':<15} {rba_auc*100:>9.2f}%  {rba_fpr*100:>9.2f}%")



  Dataset: RoadAnomaly
  Evaluated: 60 images  |  Skipped (no OoD): 0

  Method               AUPRC   FPR@95TPR
  --------------------------------------
  MSP                 81.69%      10.62%
  Entropy             73.26%      10.40%
  MaxLogit            81.86%      10.55%
  RbA                 82.13%      10.24%

  Dataset: RoadAnomaly21
  Evaluated: 10 images  |  Skipped (no OoD): 0

  Method               AUPRC   FPR@95TPR
  --------------------------------------
  MSP                 89.28%       6.02%
  Entropy             84.86%       5.53%
  MaxLogit            89.57%       5.88%
  RbA                 89.90%       4.60%

  Dataset: RoadObstacle21
  Evaluated: 30 images  |  Skipped (no OoD): 0

  Method               AUPRC   FPR@95TPR
  --------------------------------------
  MSP                 98.60%       0.02%
  Entropy             98.54%       0.02%
  MaxLogit            98.62%       0.02%
  RbA                 98.77%       0.02%

  Dataset: fs_static
  Evaluated: 20 ima